# Import packages

In [49]:
import cv2
from keras.utils import load_img
from keras.saving import load_model
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from skimage.measure import regionprops, regionprops_table
from tqdm import trange, tqdm

import segmenteverygrain as seg
import segmenteverygrain.interactions as si
import os

# %matplotlib qt

## Load models

Maybe you need to clone first SAM2. Then add the yaml file in the bottom of the next chunk

In [52]:
# UNET model
unet = load_model(
    "../models/seg_model.keras",
    custom_objects={"weighted_crossentropy": seg.weighted_crossentropy},
)

# Auto-detect device: CUDA for NVIDIA, MPS for Apple Silicon, CPU as fallback
import torch
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# SAM 2.1 checkpoints. Download from:
# https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
sam = build_sam2(r"C:\Users\leoni\sam2\sam2\configs\sam2.1\sam2.1_hiera_l.yaml", r"C:\Users\leoni\segmenteverygrain\models\sam2.1_hiera_large.pt", device=device)


## Set your directories

In [65]:
dir = os.chdir("C:/Users/leoni/kirgis_repo/segmenteverygrain/leonieexamples")
dir = os.getcwd()
os.makedirs("ouputgpkg", exist_ok=True)
os.makedirs("inputtiles", exist_ok=True)
os.makedirs("plots", exist_ok=True)


inputtiledir = os.path.join(dir, "inputtiles/")
ouputgpkg = os.path.join(dir, "ouputgpkg/")

plotdir = os.path.join(dir, "plots/")

## Run segmentation

Collect tiles

In [66]:
import glob
tiles = sorted(glob.glob(os.path.join(inputtiledir, "*.tif")))
print(f"Found {len(tiles)} tiles")

Found 1 tiles


In [67]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img


# --- collect tiles ---
tiles = sorted(glob.glob(os.path.join(inputtiledir, "*.tif")))
print(f"Found {len(tiles)} tiles")

# --- loop ---
for i, fname in enumerate(tiles, start=1):
    tile_id = os.path.splitext(os.path.basename(fname))[0]
    print(f"[{i}/{len(tiles)}] {tile_id}")

    # load + predict
    image = np.array(load_img(fname))
    image_pred = seg.predict_image(image, unet, I=256)

    # prompts (Unet -> points)
    labels, coords = seg.label_grains(image, image_pred, dbs_max_dist=10.0)

    # SAM segmentation
    all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
        sam,
        image,
        image_pred,
        coords,
        labels,
        min_area=35.0,
        plot_image=True,
        remove_edge_grains=True,
        remove_large_objects=True,
    )
    figname =os.path.join(plotdir, f"{tile_id}.png")

    fig.savefig(figname, dpi = 200, bbox_inches="tight")
    plt.close(fig)
    print("Saved the Plot")
    print("done with segmentation")






Found 1 tiles
[1/1] tile_r000008_c000032
segmenting image tiles...


100%|██████████| 3/3 [00:01<00:00,  2.49it/s]


creating masks using SAM...


100%|██████████| 290/290 [00:21<00:00, 13.49it/s]


finding overlapping polygons...


...

finding best polygons...


100%|██████████| 58/58 [00:01<00:00, 30.33it/s]


creating labeled image...


100%|██████████| 95/95 [00:00<00:00, 231.17it/s]


Saved the Plot
done with segmentation
